# Data Loading + Feature Engineering (V5)

Pipeline for the V5 dataset with frame stacking:

1. Drop formation one-hot — 
2. Yaw cos/sin transform — replace raw yaw_error with cos/sin in both frames
3. Z-score normalize inputs — computed from train split only, excluding cos/sin features
4. Max-abs scale labels — bound to [-1, 1] for PPO readiness (99th percentile for yaw)
5. Save normalization stats — for live inference in DAgger/PPO

## 1. Load dataset splits

In [3]:
from pathlib import Path
import json
import torch
import numpy as np
from torch_geometric.data import Data, InMemoryDataset

TRAIN_PATH = r"../../../datasets/setpoint_mixed_V5_mixed_formations_train.pt"
VAL_PATH   = r"../../../datasets/setpoint_mixed_V5_mixed_formations_val.pt"
TEST_PATH  = r"../../../datasets/setpoint_mixed_V5_mixed_formations_test.pt"


class SplitDataset(InMemoryDataset):
    def __init__(self, split_path):
        split_path = Path(split_path).expanduser().resolve()
        if not split_path.exists():
            raise FileNotFoundError(f"Dataset not found: {split_path}")
        payload = torch.load(split_path, weights_only=False)
        self.split_name = payload.get("split_name", "unknown")
        self.formation_names = payload.get("formation_names", [])
        super().__init__(root="")
        self.data = payload["data"]
        self.slices = payload["slices"]


train_ds = SplitDataset(TRAIN_PATH)
val_ds   = SplitDataset(VAL_PATH)
test_ds  = SplitDataset(TEST_PATH)

print(f"train graphs: {len(train_ds)}")
print(f"val graphs  : {len(val_ds)}")
print(f"test graphs : {len(test_ds)}")
print(f"formations  : {train_ds.formation_names}")

sample = train_ds[0]
print(f"\nRaw sample:")
print(f"  x shape        : {tuple(sample.x.shape)}")
print(f"  edge_attr shape: {tuple(sample.edge_attr.shape)}")
print(f"  target shape   : {tuple(sample.target.shape)}")

train graphs: 15525
val graphs  : 2104
test graphs : 2277
formations  : ('a', 'rectangle', 'triangle', 'random_cloud')

Raw sample:
  x shape        : (17, 64)
  edge_attr shape: (130, 7)
  target shape   : (17, 4)


## 2. Feature layout constants


```

In [4]:
# Layout of a SINGLE frame (before stacking)
SINGLE_FRAME_DIM_RAW = 32     
N_FORMATION = 4              
YAW_IDX_IN_FRAME = 25           
FORMATION_SLICE = slice(28, 32)    


#Edge feature layout 
EDGE_DIM = 7  # 

print(f"Raw single frame dim : {SINGLE_FRAME_DIM_RAW}")
print(f"Raw stacked dim      : {SINGLE_FRAME_DIM_RAW * 2}")
print(f"Processed frame dim  : {SINGLE_FRAME_DIM_RAW - N_FORMATION + 1}")
print(f"Processed stacked dim: {(SINGLE_FRAME_DIM_RAW - N_FORMATION + 1) * 2}")
print(f"Edge dim             : {EDGE_DIM}")

Raw single frame dim : 32
Raw stacked dim      : 64
Processed frame dim  : 29
Processed stacked dim: 58
Edge dim             : 7


## 3. Feature engineering transform

In [5]:
def process_frame(frame: torch.Tensor) -> torch.Tensor:
    """
    Process a single frame (N_drones, 32):
    1. Drop formation one-hot (last 4 cols)
    2. Replace raw yaw_error with cos(yaw), sin(yaw)

    Returns: (N_drones, 29)
    """
    # Extract parts
    before_yaw = frame[:, :YAW_IDX_IN_FRAME]        # cols 0..24  (25 dims)
    raw_yaw    = frame[:, YAW_IDX_IN_FRAME]          # col  25     (1 dim)
    after_yaw  = frame[:, YAW_IDX_IN_FRAME + 1:28]   # cols 26..27 (2 dims: floor, ceiling)
    # cols 28..31 = formation one-hot → DROPPED

    # Cos/sin transform
    cos_yaw = torch.cos(raw_yaw).unsqueeze(1)  # (N, 1)
    sin_yaw = torch.sin(raw_yaw).unsqueeze(1)  # (N, 1)

    # Reassemble: [before_yaw(25), cos_yaw(1), sin_yaw(1), floor(1), ceiling(1)] = 29
    return torch.cat([before_yaw, cos_yaw, sin_yaw, after_yaw], dim=1)


def engineer_features(x: torch.Tensor) -> torch.Tensor:
    """
    Apply feature engineering to the full stacked node features.

    Input:  (N, 64) = [frame_t0 (32), frame_t1 (32)]
    Output: (N, 58) = [processed_t0 (29), processed_t1 (29)]
    """
    frame_t0 = x[:, :SINGLE_FRAME_DIM_RAW]
    frame_t1 = x[:, SINGLE_FRAME_DIM_RAW:]

    proc_t0 = process_frame(frame_t0)
    proc_t1 = process_frame(frame_t1)

    return torch.cat([proc_t0, proc_t1], dim=1)


# ── Indices of cos/sin features in the PROCESSED 58-dim vector ──
# Frame t0: cos_yaw at index 25, sin_yaw at index 26
# Frame t1: cos_yaw at index 25+29=54, sin_yaw at index 26+29=55
PROCESSED_FRAME_DIM = 29
COS_SIN_INDICES = [25, 26, 25 + PROCESSED_FRAME_DIM, 26 + PROCESSED_FRAME_DIM]
print(f"Cos/sin feature indices (excluded from Z-score): {COS_SIN_INDICES}")

Cos/sin feature indices (excluded from Z-score): [25, 26, 54, 55]


## 4. Normalizer class

In [6]:
class DatasetNormalizer:
    """
    Computes and applies normalization statistics.

    - x: Z-score (excluding cos/sin features)
    - edge_attr: Z-score
    - target (y): max-abs scaling to [-1, 1] (99th percentile for yaw)
    """

    def __init__(self, x_mean, x_std, edge_mean, edge_std, y_scale,
                 cos_sin_indices, input_dim, edge_dim, target_dim):
        self.x_mean = x_mean
        self.x_std = x_std
        self.edge_mean = edge_mean
        self.edge_std = edge_std
        self.y_scale = y_scale
        self.cos_sin_indices = cos_sin_indices
        self.input_dim = input_dim
        self.edge_dim = edge_dim
        self.target_dim = target_dim

    @classmethod
    def fit(cls, train_ds):
        """
        Compute normalization stats from the training split ONLY.
        Must be called AFTER feature engineering (process_frame / engineer_features).
        """
        # ── Step 1: Apply feature engineering to all training data ──
        all_x = engineer_features(train_ds.data.x)
        input_dim = all_x.shape[1]

        # ── Step 2: Z-score stats for x (excluding cos/sin) ──
        x_mean = all_x.mean(dim=0)
        x_std = all_x.std(dim=0).clamp(min=1e-6)

        # Set cos/sin features to identity normalization (mean=0, std=1)
        for idx in COS_SIN_INDICES:
            x_mean[idx] = 0.0
            x_std[idx] = 1.0

        # ── Step 3: Z-score stats for edge_attr ──
        all_edge = train_ds.data.edge_attr
        edge_dim = all_edge.shape[1] if all_edge is not None and all_edge.numel() > 0 else EDGE_DIM
        if all_edge is not None and all_edge.numel() > 0:
            edge_mean = all_edge.mean(dim=0)
            edge_std = all_edge.std(dim=0).clamp(min=1e-6)
        else:
            edge_mean = torch.zeros(edge_dim)
            edge_std = torch.ones(edge_dim)

        # ── Step 4: Max-abs scaling for labels ──
        all_y = train_ds.data.target
        target_dim = all_y.shape[1]

        # For dx, dy, dz: use true max absolute value
        y_max_abs = all_y.abs().max(dim=0).values

        # For dyaw (index 3): use 99th percentile to avoid wrapping artifacts
        yaw_abs = all_y[:, 3].abs()
        yaw_99th = torch.quantile(yaw_abs, 0.99)
        y_max_abs[3] = yaw_99th

        # Safety: clamp to avoid division by zero
        y_scale = y_max_abs.clamp(min=1e-6)

        print(f"Input dim (after engineering): {input_dim}")
        print(f"Edge dim: {edge_dim}")
        print(f"Target dim: {target_dim}")
        print(f"\ny_scale (max-abs, 99th for yaw): {y_scale}")
        print(f"Cos/sin indices (excluded from Z-score): {COS_SIN_INDICES}")

        return cls(
            x_mean=x_mean, x_std=x_std,
            edge_mean=edge_mean, edge_std=edge_std,
            y_scale=y_scale,
            cos_sin_indices=COS_SIN_INDICES,
            input_dim=input_dim,
            edge_dim=edge_dim,
            target_dim=target_dim,
        )

    def transform_graph(self, graph: Data) -> Data:
        """Apply feature engineering + normalization to a single graph."""
        # Feature engineering
        x_eng = engineer_features(graph.x)

        # Z-score normalize x
        x_norm = (x_eng - self.x_mean) / self.x_std

        # Z-score normalize edge_attr
        if graph.edge_attr is not None and graph.edge_attr.numel() > 0:
            edge_norm = (graph.edge_attr - self.edge_mean) / self.edge_std
        else:
            edge_norm = graph.edge_attr

        # Max-abs scale labels, clamp to [-1, 1]
        y_norm = torch.clamp(graph.target / self.y_scale, -1.0, 1.0)

        return Data(
            x=x_norm,
            target=y_norm,
            edge_index=graph.edge_index,
            edge_attr=edge_norm,
            pos=graph.pos,
            formation_id=graph.formation_id,
            episode_id=graph.episode_id,
            step_idx=graph.step_idx,
            num_drones=graph.num_drones,
        )

    def save(self, path):
        """Save normalization stats as a .pt file for inference."""
        torch.save({
            "x_mean": self.x_mean,
            "x_std": self.x_std,
            "edge_mean": self.edge_mean,
            "edge_std": self.edge_std,
            "y_scale": self.y_scale,
            "cos_sin_indices": self.cos_sin_indices,
            "input_dim": self.input_dim,
            "edge_dim": self.edge_dim,
            "target_dim": self.target_dim,
        }, path)
        print(f"Saved normalization stats: {path}")

    @classmethod
    def load(cls, path):
        """Load normalization stats from a .pt file."""
        d = torch.load(path, weights_only=False)
        return cls(**d)

## 5. Fit normalizer on training data

In [7]:
normalizer = DatasetNormalizer.fit(train_ds)

# ── Verify on a sample graph ──
raw = train_ds[0]
norm = normalizer.transform_graph(raw)

print(f"\n{'='*50}")
print(f"Sample graph before/after:")
print(f"  raw x         : {tuple(raw.x.shape)}")
print(f"  processed x   : {tuple(norm.x.shape)}")
print(f"  raw edge_attr : {tuple(raw.edge_attr.shape) if raw.edge_attr is not None else 'None'}")
print(f"  norm edge_attr: {tuple(norm.edge_attr.shape) if norm.edge_attr is not None else 'None'}")
print(f"  raw target    : {tuple(raw.target.shape)}")
print(f"  norm target   : {tuple(norm.target.shape)}")

# ── Sanity checks ──
print(f"\nSanity checks:")
print(f"  norm x mean ≈ 0 : {norm.x.mean(dim=0).abs().max().item():.6f}")
print(f"  norm x std  ≈ 1 : {norm.x.std(dim=0).mean().item():.4f}")
if norm.target.numel() > 0:
    print(f"  norm target range: [{norm.target.min().item():.4f}, {norm.target.max().item():.4f}]")

C:\Users\aimen\AppData\Local\Temp\ipykernel_7412\437844780.py:29: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache now by removing all elements in `dataset._data_list`. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  all_x = engineer_features(train_ds.data.x)


Input dim (after engineering): 58
Edge dim: 7
Target dim: 4

y_scale (max-abs, 99th for yaw): tensor([0.0393, 0.0323, 0.0141, 0.0237])
Cos/sin indices (excluded from Z-score): [25, 26, 54, 55]

Sample graph before/after:
  raw x         : (17, 64)
  processed x   : (17, 58)
  raw edge_attr : (130, 7)
  norm edge_attr: (130, 7)
  raw target    : (17, 4)
  norm target   : (17, 4)

Sanity checks:
  norm x mean ≈ 0 : 7.064766
  norm x std  ≈ 1 : 0.1257
  norm target range: [-0.3845, 0.3743]


C:\Users\aimen\AppData\Local\Temp\ipykernel_7412\437844780.py:42: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  all_edge = train_ds.data.edge_attr
C:\Users\aimen\AppData\Local\Temp\ipykernel_7412\437844780.py:52: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  all_y = train_ds.data.target


## 6. Save normalization stats

In [8]:
RESULTS_DIR = Path("../results").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

stats_path = RESULTS_DIR / "normalization_stats.pt"
normalizer.save(stats_path)

# ── Verify reload ──
loaded = DatasetNormalizer.load(stats_path)
print(f"Reload check — input_dim: {loaded.input_dim}, edge_dim: {loaded.edge_dim}, target_dim: {loaded.target_dim}")
print(f"y_scale: {loaded.y_scale}")

Saved normalization stats: C:\Users\aimen\OneDrive\Desktop\gnn_drone_project\gnn\setpoint_prediction2\results\normalization_stats.pt
Reload check — input_dim: 58, edge_dim: 7, target_dim: 4
y_scale: tensor([0.0393, 0.0323, 0.0141, 0.0237])
